# Lecture 22: Adaptive Experimentation (Multi-Armed Bandits)

**Three worlds, one piece of machinery.** A multi-armed bandit is an adaptive experiment: K actions, T rounds, allocate more of T to actions that look good as evidence accumulates. Today's readings cover three worlds where the same machinery gets used with different operational constraints.

**Required readings (pick one per world):**

- **Politics.** Lee & Green (2024). *Discovering optimal ballot wording using adaptive survey design.* Research & Politics 11(2). 11 ballot-wording arms, 2,168 respondents, 5 Western US states. Harvard Dataverse doi:10.7910/DVN/O273OW. Same Green as the 2021 Offer-Westort / Coppock / Green paper, applied to a cleaner problem.
- **Operations.** Che & Namkoong (2023). *Adaptive Experimentation at Scale: A Computational Framework for Flexible Batches.* arXiv 2303.11582. Real experiments only get a handful of reallocation opportunities (batches); they derive a Gaussian sequential approximation that plans batched allocations via dynamic programming. Their Residual Horizon Optimization (RHO) beats Thompson sampling when SNR is low and batch count is small.
- **mHealth.** Nahum-Shani & Murphy (2026). *Just-in-Time Adaptive Interventions: Where Are We Now and What Is Next?* Annual Review of Psychology. The state-of-the-field review of JITAIs - adaptive behavioral interventions that decide *when* and *what* to deliver to each person based on real-time context (smoking cessation, physical activity, cardiac rehab).

**What this notebook covers:**

1. Sections 1-5 (SIMULATED): epsilon-greedy, UCB, and Thompson sampling on a 5-arm toy problem - the core algorithms every bandit paper builds on.
2. Section 6 (REAL): replication of Offer-Westort, Coppock & Green (2021) Study 1 from Harvard Dataverse (`doi:10.7910/DVN/CMUHBU`). 8-arm Thompson sampling on right-to-work messaging, 10 batches, 1000 MTurk participants. This is old Green - the 2021 paper. Lee & Green (2024) is new Green and uses the same machinery with 11 arms.
3. Reflection questions (answer in the Google Form).

**Data mix:** Sections 1-5 use SIMULATED persuasion rates. Section 6 uses REAL data from Harvard Dataverse.


## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)
print('✓ Ready to run bandits simulations')

## The Setup: 5 Message Variants

You're running a political campaign and have 5 different message framings:

1. **Economic message**: Focus on job creation
2. **Healthcare message**: Focus on affordability
3. **Education message**: Focus on opportunity
4. **Climate message**: Focus on future
5. **Unity message**: Focus on bringing people together

Each message has a true persuasion rate (unknown to you). Your goal: learn which is best, but ALSO show the best ones more often.


In [ ]:
# True persuasion rates (you don't know these during the campaign)
true_rates = np.array([0.10, 0.12, 0.08, 0.07, 0.11])  # Economic, Healthcare, Education, Climate, Unity
message_names = ['Economic', 'Healthcare', 'Education', 'Climate', 'Unity']

print('True persuasion rates (secret):')
for i, (name, rate) in enumerate(zip(message_names, true_rates)):
    print(f'  {i+1}. {name}: {rate:.1%}')
print(f'\nBest message: {message_names[np.argmax(true_rates)]} ({np.max(true_rates):.1%})')

## Algorithm 1: Epsilon-Greedy

Strategy: 90% of the time, show the best message so far. 10% of the time, try a random message.


In [ ]:
def epsilon_greedy(true_rates, n_rounds=2000, epsilon=0.1):
    """Epsilon-greedy bandit."""
    n_arms = len(true_rates)
    clicks = np.zeros(n_arms)
    impressions = np.zeros(n_arms)
    allocation = []
    rewards = []
    
    for t in range(n_rounds):
        # Decide which message to show
        if np.random.rand() < epsilon:  # Explore
            arm = np.random.randint(n_arms)
        else:  # Exploit
            obs_rates = clicks / np.maximum(impressions, 1)
            arm = np.argmax(obs_rates)
        
        # Show the message, get reward
        impressions[arm] += 1
        reward = int(np.random.rand() < true_rates[arm])
        clicks[arm] += reward
        allocation.append(arm)
        rewards.append(reward)
    
    return clicks, impressions, allocation, rewards

clicks_eg, impr_eg, alloc_eg, rewards_eg = epsilon_greedy(true_rates)

print('\n=== EPSILON-GREEDY RESULTS ===')
for i, name in enumerate(message_names):
    obs_rate = clicks_eg[i] / impr_eg[i] if impr_eg[i] > 0 else 0
    print(f'{name:12} | Impressions: {int(impr_eg[i]):4} | Clicks: {int(clicks_eg[i]):3} | Rate: {obs_rate:.1%}')

print(f'\nTotal persuaded: {int(np.sum(clicks_eg))}')

## Algorithm 2: Upper Confidence Bound (UCB)

Strategy: For each message, compute observed rate + uncertainty bonus. Show the message with the highest total.


In [ ]:
def ucb_bandit(true_rates, n_rounds=2000, c=0.3):
    """UCB1 bandit. The `c` parameter controls exploration:
    smaller c = more exploitation."""
    n_arms = len(true_rates)
    clicks = np.zeros(n_arms)
    impressions = np.zeros(n_arms)
    allocation = []
    rewards = []
    
    # Warm-start: pull each arm once
    for a in range(n_arms):
        impressions[a] = 1
        reward = int(np.random.rand() < true_rates[a])
        clicks[a] = reward
        allocation.append(a)
        rewards.append(reward)
    
    for t in range(n_arms, n_rounds):
        obs_rates = clicks / impressions
        uncertainty = c * np.sqrt(np.log(t + 1) / impressions)
        ucb_values = obs_rates + uncertainty
        arm = int(np.argmax(ucb_values))
        
        impressions[arm] += 1
        reward = int(np.random.rand() < true_rates[arm])
        clicks[arm] += reward
        allocation.append(arm)
        rewards.append(reward)
    
    return clicks, impressions, allocation, rewards

clicks_ucb, impr_ucb, alloc_ucb, rewards_ucb = ucb_bandit(true_rates)

print('\n=== UCB RESULTS (c=0.3) ===')
for i, name in enumerate(message_names):
    obs_rate = clicks_ucb[i] / impr_ucb[i] if impr_ucb[i] > 0 else 0
    print(f'{name:12} | Impressions: {int(impr_ucb[i]):4} | Clicks: {int(clicks_ucb[i]):3} | Rate: {obs_rate:.1%}')

print(f'\nTotal persuaded: {int(np.sum(clicks_ucb))}')

## Algorithm 3: Thompson Sampling

Strategy: For each message, maintain a belief distribution (a Beta for Bernoulli rewards). Each round, sample one value from each arm's distribution and show whichever sampled highest. As evidence accumulates, the distributions tighten around the true rates; uncertain arms get explored in proportion to the probability they might actually be best.


In [ ]:
def thompson_sampling(true_rates, n_rounds=2000):
    """Thompson sampling with Beta-Bernoulli model."""
    n_arms = len(true_rates)
    successes = np.ones(n_arms)  # Beta prior: successes (α)
    failures = np.ones(n_arms)   # Beta prior: failures (β)
    allocation = []
    rewards = []
    
    for t in range(n_rounds):
        # Sample from each arm's Beta distribution
        samples = np.random.beta(successes, failures)
        arm = np.argmax(samples)
        
        # Get reward
        reward = int(np.random.rand() < true_rates[arm])
        if reward:
            successes[arm] += 1
        else:
            failures[arm] += 1
        
        allocation.append(arm)
        rewards.append(reward)
    
    clicks = successes - 1  # Remove the prior
    impressions = successes + failures - 2
    return clicks, impressions, allocation, rewards

clicks_ts, impr_ts, alloc_ts, rewards_ts = thompson_sampling(true_rates)

print('\n=== THOMPSON SAMPLING RESULTS ===')
for i, name in enumerate(message_names):
    obs_rate = clicks_ts[i] / impr_ts[i] if impr_ts[i] > 0 else 0
    print(f'{name:12} | Impressions: {int(impr_ts[i]):4} | Clicks: {int(clicks_ts[i]):3} | Rate: {obs_rate:.1%}')

print(f'\nTotal persuaded: {int(np.sum(clicks_ts))}')

## Comparison: Which algorithm wins?

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Epsilon-Greedy
alloc_counts_eg = np.bincount(alloc_eg, minlength=5)
axes[0].bar(message_names, alloc_counts_eg, color='steelblue')
axes[0].set_title('Epsilon-Greedy\nAllocation')
axes[0].set_ylabel('Impressions')
axes[0].tick_params(axis='x', rotation=45)

# UCB
alloc_counts_ucb = np.bincount(alloc_ucb, minlength=5)
axes[1].bar(message_names, alloc_counts_ucb, color='darkgreen')
axes[1].set_title('UCB\nAllocation')
axes[1].set_ylabel('Impressions')
axes[1].tick_params(axis='x', rotation=45)

# Thompson Sampling
alloc_counts_ts = np.bincount(alloc_ts, minlength=5)
axes[2].bar(message_names, alloc_counts_ts, color='darkred')
axes[2].set_title('Thompson Sampling\nAllocation')
axes[2].set_ylabel('Impressions')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Total reward comparison
print('\n=== TOTAL PERFORMANCE ===')
print(f'Epsilon-Greedy:    {int(np.sum(rewards_eg))} persuaded')
print(f'UCB:               {int(np.sum(rewards_ucb))} persuaded')
print(f'Thompson Sampling: {int(np.sum(rewards_ts))} persuaded')
print(f'\nIf you showed only the best message: {int(2000 * np.max(true_rates))} persuaded')

---
## Visualization 1: Epsilon strategies over time

Three ways to schedule the exploration rate in epsilon-greedy:

1. **Constant**: $\epsilon = 0.1$ forever
2. **Step**: $\epsilon = 1$ for $t < 100$, then $\epsilon = 0$
3. **Decaying**: $\epsilon = 1/(t+1)$

Which one wins?

In [ ]:
def epsilon_variants(true_rates, n_rounds=2000):
    n_arms = len(true_rates)
    
    def run(epsilon_fn):
        clicks = np.zeros(n_arms)
        impressions = np.zeros(n_arms)
        cum_reward = []
        total = 0
        for t in range(n_rounds):
            eps = epsilon_fn(t)
            if np.random.rand() < eps or impressions.min() == 0:
                arm = np.random.randint(n_arms)
            else:
                rates = clicks / np.maximum(impressions, 1)
                arm = int(np.argmax(rates))
            impressions[arm] += 1
            reward = int(np.random.rand() < true_rates[arm])
            clicks[arm] += reward
            total += reward
            cum_reward.append(total)
        return cum_reward
    
    # Average over 100 sims to smooth the plot
    n_sims = 100
    strategies = {
        'Constant (eps=0.1)': lambda t: 0.1,
        'Step (eps=1 if t<100 else 0)': lambda t: 1.0 if t < 100 else 0.0,
        'Decaying (eps=1/(t+1))': lambda t: 1.0 / (t + 1),
        'Fast decay (eps=1/(t+1)^2)': lambda t: 1.0 / (t + 1)**2,
    }
    results = {name: np.zeros(n_rounds) for name in strategies}
    for _ in range(n_sims):
        for name, fn in strategies.items():
            results[name] += np.array(run(fn)) / n_sims
    return results

eps_results = epsilon_variants(true_rates)

plt.figure(figsize=(11, 5))
colors = {'Constant (eps=0.1)': 'steelblue', 'Step (eps=1 if t<100 else 0)': 'darkorange', 'Decaying (eps=1/(t+1))': 'darkgreen', 'Fast decay (eps=1/(t+1)^2)': 'crimson'}
for name, curve in eps_results.items():
    plt.plot(curve, label=name, color=colors[name])
plt.plot(np.arange(2000) * np.max(true_rates), '--', color='gray', alpha=0.5, label='Best possible')
plt.xlabel('Round')
plt.ylabel('Cumulative persuaded')
plt.title('Epsilon schedules: constant vs step vs decaying')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print('Final totals (avg over 100 sims):')
for name, curve in eps_results.items():
    print(f'  {name}: {curve[-1]:.0f}')

---
## Visualization 2: Thompson sampling posteriors over time

Thompson sampling maintains a Beta distribution for each arm. Early on, they're all wide. As evidence accumulates, they tighten around the true rates.

Watch them converge.

In [ ]:
from scipy.stats import beta

def thompson_with_posteriors(true_rates, n_rounds=2000, snapshot_rounds=(10, 100, 500, 2000)):
    n_arms = len(true_rates)
    alpha = np.ones(n_arms)
    beta_params = np.ones(n_arms)
    snapshots = {}
    
    for t in range(1, n_rounds + 1):
        samples = np.random.beta(alpha, beta_params)
        arm = int(np.argmax(samples))
        reward = int(np.random.rand() < true_rates[arm])
        if reward:
            alpha[arm] += 1
        else:
            beta_params[arm] += 1
        if t in snapshot_rounds:
            snapshots[t] = (alpha.copy(), beta_params.copy())
    
    return snapshots

snapshots = thompson_with_posteriors(true_rates)

fig, axes = plt.subplots(1, 4, figsize=(18, 4), sharey=True)
x = np.linspace(0, 0.25, 400)
colors = plt.cm.tab10(np.linspace(0, 1, len(true_rates)))

for ax, (round_num, (a, b)) in zip(axes, snapshots.items()):
    for i in range(len(true_rates)):
        pdf = beta.pdf(x, a[i], b[i])
        ax.plot(x, pdf, color=colors[i], label=f'{message_names[i]}')
        ax.axvline(true_rates[i], color=colors[i], linestyle='--', alpha=0.5)
    ax.set_title(f'After {round_num} rounds')
    ax.set_xlabel('True rate')
    ax.grid(alpha=0.3)

axes[0].set_ylabel('Posterior density')
axes[0].legend(loc='upper right', fontsize=8)
plt.suptitle('Thompson sampling: posterior beliefs tighten over time\n(dashed lines = true rates)')
plt.tight_layout()
plt.show()

---
## Predict before you run

Before you run the next section on Offer-Westort's real Dataverse data:

Thompson sampling ran on an 8-arm bandit for 10 batches (1000 MTurk participants total). One arm won.

**Your prediction:** In **batch 1** (early, no information yet), what fraction of impressions went to the winning arm? By **batch 5** (some learning), what fraction went to the winning arm?

Write your guesses down, then run the next cells and check.

*(Hint: there are 8 arms. Uniform allocation would be 12.5% per arm.)*

---
## Section 6: Old Green (Offer-Westort, Coppock & Green 2021) [REAL]

**Data:** actual experiment data from Offer-Westort, Coppock & Green's (2021, AJPS) Study 1, loaded from Harvard Dataverse (`doi:10.7910/DVN/CMUHBU`). 1000 MTurk participants (not voters, as OWG acknowledge), 10 batches of ~100, 8 right-to-work messaging arms. The authors ran a Thompson-sampling bandit: each batch's allocation depended on posteriors updated from prior batches' outcomes.

Sections 1-5 showed Thompson sampling on simulated data. This section applies it to real survey data. You should see Proposal 4 (BM) concentrating traffic as the algorithm learns it is the winner.

**What Lee & Green (2024) did differently.** Same Don Green, three years later, 11 arms instead of 8, 2,168 respondents instead of 1000, 5 Western US states instead of MTurk, and the question was not "which right-to-work framing persuades" but "which phrasing of a ballot measure maximizes yes votes." The machinery is identical; the application is cleaner. Harvard Dataverse `doi:10.7910/DVN/O273OW` has the replication data if you want to rerun the allocation plot with 11 arms instead of 8.


In [ ]:
# [REAL] Load Offer-Westort-Green Study 1 data from Harvard Dataverse
# 8-arm bandit on right-to-work messaging, 10 batches, 1000 MTurk participants
data_url = 'https://raw.githubusercontent.com/Persuasion-at-Scale/effects-to-decisions/main/data/offer-westort-2021-study1.csv'
ow = pd.read_csv(data_url)

print(f"[REAL] Loaded: {len(ow)} participants, {ow['batch'].nunique()} batches, "
      f"{ow['Z_rtw'].nunique()} RTW arms")
print(f"Columns: {list(ow.columns)}")
print()

# Arm-level success rates (binary agreement outcome)
arm_stats = ow.groupby('Z_rtw').agg(
    n=('Y_rtw', 'size'),
    success_rate=('Y_rtw', 'mean')
).sort_values('n', ascending=False)
print("Arm allocation + success rate (sorted by final allocation):")
print(arm_stats.round(3))

In [ ]:
# Define colors (needed by plot)
GREEN = '#2ca02c'
GRAY = '#888888'
RED = '#d62728'

# [REAL] Plot fraction of each batch assigned to each arm
alloc = ow.groupby(['batch', 'Z_rtw']).size().unstack(fill_value=0)
alloc_frac = alloc.div(alloc.sum(axis=1), axis=0)

# Identify winner for highlight
winner = arm_stats.index[0]

fig, ax = plt.subplots(figsize=(11, 6))
for arm in alloc_frac.columns:
    is_winner = (arm == winner)
    ax.plot(alloc_frac.index, alloc_frac[arm],
            marker='o', linewidth=(3 if is_winner else 1.2),
            alpha=(1.0 if is_winner else 0.45),
            color=(GREEN if is_winner else GRAY),
            label=arm if is_winner else None,
            zorder=(5 if is_winner else 2))

ax.axhline(1/8, color=RED, linestyle='--', linewidth=2,
           label='Static uniform (1/8 = 0.125)')
ax.set_xlabel('Batch (each ~100 participants)', fontsize=12)
ax.set_ylabel('Fraction of batch assigned to arm', fontsize=12)
ax.set_title(
    '[REAL] Offer-Westort-Green Study 1: adaptive allocation across 10 batches\n'
    f'Thompson sampling concentrated traffic on {winner}',
    fontsize=13
)
ax.set_ylim(0, 1.0)
ax.legend(loc='center left', fontsize=10)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nBy batch 10, Thompson sampling was sending "
      f"{alloc_frac.iloc[-1][winner]:.0%} of traffic to {winner}")
print(f"Static uniform would have sent 12.5% to each arm.")
print(f"Cumulative total: {arm_stats.loc[winner,'n']}/{len(ow)} = "
      f"{arm_stats.loc[winner,'n']/len(ow):.0%} of all 1000 participants.")

**What you should see [REAL data]:** In early batches (1-3), Thompson sampling allocates participants roughly uniformly across the 8 arms (~12-25% each) because posteriors are still flat. As evidence accumulates that Proposal 4 (BM) has a higher success rate, the algorithm concentrates subsequent batches on it, reaching 80-95% of traffic by the final batches. Static uniform allocation (red dashed line at 12.5%) is oblivious to the evidence and wastes participants on losing arms.

Same total budget of 1000 participants, many more hits on the winner. That is the adaptive move Offer-Westort, Coppock, and Green advocate.

---

## Where the three worlds diverge

Sections 1-6 showed the machinery on ~1000-2000 participants. Real applications scale differently across the three worlds:

1. **Politics (Lee & Green 2024).** 11 ballot-wording arms, 2,168 respondents, 5 states, short horizon (survey panel). Dataverse doi:10.7910/DVN/O273OW. Same Thompson machinery, cleaner outcome.
2. **Operations (Che & Namkoong 2023).** Real A/B platforms only get a handful of reallocation opportunities (1-10 batches), not continuous updates. Standard Thompson sampling assumes continuous reallocation and leaves performance on the table in low-batch / low-SNR regimes. Their Residual Horizon Optimization (RHO) uses dynamic programming over a Gaussian sequential approximation to plan the full remaining budget at each reallocation, and beats Thompson when you only get a few shots.
3. **mHealth (Nahum-Shani & Murphy 2026).** Text-message nudges to cardiac patients, smoking-cessation prompts, physical activity reminders. The decision is not just *which* action (as in Sections 1-6) but *when*: deliver now, wait, or skip. JITAIs add a temporal dimension Sections 1-6 ignore.

**Next class (Apr 15): contextual bandits.** Same machinery, but the right action depends on who the person is (age, party ID, prior donations). In our notation, we stop learning a scalar per arm and start learning $f_a(X) = E[Y \mid A{=}a, X{=}x]$. Then pick $\arg\max_a f_a(X)$ per person.


## Reflection Questions (answer in the Google Form)

1. **Which algorithm persuaded the most people in Sections 1-5?** Why do you think that is? Would the ranking change if the gap between the best arm and the second-best arm were smaller?

2. **Offer-Westort, Coppock & Green (2021) found Thompson Sampling identified the winning message faster than fixed A/B.** In the simulated Section 5, how many rounds did Thompson sampling need before the best arm was pulled more than 50% of the time?

3. **Lee & Green (2024) extends OWG to 11 arms and 2,168 respondents.** More arms means more exploration is needed before any one arm wins enough evidence. In your intuition, should Lee & Green's winning arm end up with a *higher* or *lower* final allocation share than OWG's winning arm? Why?

4. **Che & Namkoong (2023) argue Thompson sampling wastes performance when you only get a few reallocation batches.** Their Residual Horizon Optimization (RHO) plans the full remaining budget via dynamic programming. In what regime is this gap biggest: many batches or few; high SNR or low; many arms or few?

5. **Nahum-Shani & Murphy (2026) note JITAIs must decide *when* to act, not just *which* action.** Name one thing a JITAI has to reason about that the simulations in Sections 1-5 completely ignore. What would you have to add to the epsilon-greedy or Thompson code above to handle it?

6. **Prediction check.** Earlier you guessed what fraction of the batch went to the winning arm at batches 1 and 5 of the Offer-Westort replication. Which guess was closer to the real allocation? What does the gap tell you about how fast Thompson sampling concentrates on a winner when one arm is clearly dominant?
